<a href="https://colab.research.google.com/github/PranjalAsthana/InfoRetrievalMethods/blob/main/BlockSortBasedIndexing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pickle

In [2]:
class BSBIIndexer:

  def __init__(self, output_dir='bsbi_blocks', max_block_size=5):
    self.output_dir = output_dir
    self.max_block_size = max_block_size  # Max number of pairs per block
    os.makedirs(self.output_dir, exist_ok=True)

    # Term/Doc mappings (typically held in memory or external maps)
    self.term_to_id = {}
    self.id_counter = 0
    self.block_counter = 0

  def get_term_id(self, term):
    """Maps a string term to a unique numerical term_id."""
    if term not in self.term_to_id:
      self.term_to_id[term] = self.id_counter
      self.id_counter += 1
    return self.term_to_id[term]

  def bsbi_invert(self, raw_token_stream):
    """Processes tokens using the BSBI strategy:

    accumulates pairs, sorts them, collapses them into postings, and dumps.
    """
    block_files = []
    # record_buffer stores raw tuples: (term_id, doc_id)
    record_buffer = []

    for term, doc_id in raw_token_stream:
      term_id = self.get_term_id(term)
      record_buffer.append((term_id, doc_id))

      # Once the pair buffer reaches the threshold, process and clear it
      if len(record_buffer) >= self.max_block_size:
        block_file = self._process_and_write_block(record_buffer)
        block_files.append(block_file)
        record_buffer = []

    # Write any remaining pairs left in the buffer
    if record_buffer:
      block_file = self._process_and_write_block(record_buffer)
      block_files.append(block_file)

    return block_files

  def _process_and_write_block(self, record_buffer):
    self.block_counter += 1
    block_filename = os.path.join(
        self.output_dir, f'bsbi_block_{self.block_counter}.bin'
    )

    # Step 1: Core BSBI step - Sort pairs globally by term_id, then doc_id
    record_buffer.sort(key=lambda pair: (pair[0], pair[1]))

    # Step 2: Parse and collapse pairs into a postings format (Inverted Index)
    block_index = {}
    for term_id, doc_id in record_buffer:
      if term_id not in block_index:
        block_index[term_id] = []

      # Avoid duplicate doc_ids within the list
      if (
          not block_index[term_id]
          or block_index[term_id][-1] != doc_id
      ):
        block_index[term_id].append(doc_id)

    # Step 3: Write the structured dictionary to disk
    with open(block_filename, 'wb') as f:
      pickle.dump(block_index, f)

    return block_filename





In [3]:
# --- Example Usage ---
def dummy_token_stream():
  # Simulating a stream of (term, doc_id) tuples from a tokenizer
  tokens = [
      ('information', 1),
      ('retrieval', 1),
      ('bsbi', 1),
      ('python', 2),
      ('information', 2),
      ('indexing', 2),
  ]
  for t in tokens:
    yield t


if __name__ == '__main__':
  # Small block size to force splitting into multiple blocks
  indexer = BSBIIndexer(output_dir='bsbi_blocks', max_block_size=4)
  written_blocks = indexer.bsbi_invert(dummy_token_stream())

  print('Generated BSBI block files:', written_blocks)
  print('Vocabulary Map:', indexer.term_to_id)

Generated BSBI block files: ['bsbi_blocks/bsbi_block_1.bin', 'bsbi_blocks/bsbi_block_2.bin']
Vocabulary Map: {'information': 0, 'retrieval': 1, 'bsbi': 2, 'python': 3, 'indexing': 4}
